## 6. RQ4 — Moderation by therapeutic area

Strata are the six top-level MeSH disease categories built by `prep_06_mesh_areas.py`.
The residual `Other` category is retained as a control level but **not interpreted** —
66% of it is MeSH-mapped outside the five named categories and 34% is unmapped.


## 1. Environment and data access

In [ ]:
# Colab: install what the base image lacks!pip install -q lifelines shap pyarrow 2>/dev/nullimport pandas as pd, numpy as np, mathpd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)print("pandas", pd.__version__)

In [ ]:
# Mount Google Drive and point at the analysis-ready extractfrom google.colab import drivedrive.mount('/content/drive')DATA = '/content/drive/MyDrive/DBA_Capstone/analysis_ready'   # adjust if neededimport os; print(sorted(os.listdir(DATA)))

### Local alternativeIf running on the Mac rather than Colab, replace the mount cell with:```pythonDATA = '/Volumes/Big256/DBA - Project Datasets/09_analysis_ready'```

## 2. Load and build the analytic table

In [ ]:
trials = pd.read_parquet(f"{DATA}/ctgov_trials.parquet")ob     = pd.read_parquet(f"{DATA}/orangebook_ip.parquet")reg    = pd.read_parquet(f"{DATA}/drugsatfda_regulatory.parquet")print("trials", trials.shape, "| orangebook", ob.shape, "| regulatory", reg.shape)# Study frame: Phase II/III drug trials, start 2008-2023, resolved outcomeframe = trials[(trials.start_year.between(2008, 2023)) & (trials.is_resolved == 1)].copy()print(f"analytic frame N = {len(frame):,}")print(f"termination rate  = {frame.is_terminated.mean():.4f}")print(f"industry subset   = {int(frame.is_industry.sum()):,}")

In [ ]:
# Sponsor-level history computed with PAST-ONLY information (prevents temporal leakage)frame = frame.sort_values("start_date")g = frame.groupby("lead_sponsor")frame["sponsor_prior_trials"] = g.cumcount()frame["sponsor_prior_terminations"] = (    g["is_terminated"].cumsum() - frame["is_terminated"])frame["sponsor_prior_completion_rate"] = np.where(    frame.sponsor_prior_trials > 0,    1 - frame.sponsor_prior_terminations / frame.sponsor_prior_trials, np.nan)frame[["lead_sponsor","start_date","sponsor_prior_trials","sponsor_prior_completion_rate"]].tail(5)

## 3. RQ1 — Do termination rates differ by sponsor IP strength?

In [ ]:
from statsmodels.stats.proportion import proportions_ztestfrom scipy.stats import chi2_contingency# Proxy for IP strength until the BigQuery patent pull lands:# sponsor scale (prior trial count) median split. Replace with assignee_portfolio_size.med = frame.sponsor_prior_trials.median()frame["ip_strong"] = (frame.sponsor_prior_trials > med).astype(int)ct = pd.crosstab(frame.ip_strong, frame.is_terminated)print(ct)cnt = [ct.loc[1,1], ct.loc[0,1]]; nobs = [ct.loc[1].sum(), ct.loc[0].sum()]z, p = proportions_ztest(cnt, nobs)chi2, pchi, dof, _ = chi2_contingency(ct)v = math.sqrt(chi2 / (ct.values.sum() * min(ct.shape[0]-1, ct.shape[1]-1)))print(f"\nstrong-IP termination = {cnt[0]/nobs[0]:.4f}   weak-IP = {cnt[1]/nobs[1]:.4f}")print(f"z = {z:.4f}, p = {p:.3e}")print(f"chi2 = {chi2:.2f}, p = {pchi:.3e}, Cramer's V = {v:.4f}")

## 4. RQ2 — Predicting termination

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import HistGradientBoostingClassifierfrom sklearn.metrics import roc_auc_score, brier_score_loss, classification_reportfrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import make_pipelineFEATS = ["enrollment","is_randomized","who_masked_n","n_arms","n_locations","n_countries",         "is_multinational","elig_criteria_lines","n_primary_outcomes","n_secondary_outcomes",         "is_phase3","is_industry","n_collaborators","n_interventions",         "sponsor_prior_trials","sponsor_prior_completion_rate","start_year"]# TEMPORAL split - reflects real deployment, not random shufflingtr = frame[frame.start_year <= 2019]te = frame[frame.start_year >= 2020]Xtr, ytr = tr[FEATS].fillna(-1), tr.is_terminatedXte, yte = te[FEATS].fillna(-1), te.is_terminatedprint(f"train {len(tr):,} (2008-2019)   test {len(te):,} (2020-2023)")lr = make_pipeline(StandardScaler(),      LogisticRegression(penalty="l1", solver="liblinear", C=0.5,                         class_weight="balanced", max_iter=2000)).fit(Xtr, ytr)gb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,                                    class_weight="balanced", random_state=42).fit(Xtr, ytr)for name, m in [("L1 logistic", lr), ("Gradient boosting", gb)]:    pr = m.predict_proba(Xte)[:,1]    print(f"{name:20s} AUC={roc_auc_score(yte,pr):.4f}  Brier={brier_score_loss(yte,pr):.4f}")

In [ ]:
# SHAP attribution for the boosted modelimport shapexpl = shap.TreeExplainer(gb)sv = expl.shap_values(Xte.sample(min(2000, len(Xte)), random_state=0))shap.summary_plot(sv, Xte.sample(min(2000, len(Xte)), random_state=0), max_display=15)

## 5. RQ3 — Time to primary completion

In [ ]:
from lifelines import KaplanMeierFitter, CoxPHFittersurv = frame[frame.months_to_primary_completion.notna() &             (frame.months_to_primary_completion > 0)].copy()surv["event"] = (surv.is_terminated == 0).astype(int)   # event = reached completionprint(f"survival sample N = {len(surv):,}  events = {int(surv.event.sum()):,}")kmf = KaplanMeierFitter()ax = Nonefor lvl, lab in [(1,"Strong IP"), (0,"Weak IP")]:    s = surv[surv.ip_strong == lvl]    kmf.fit(s.months_to_primary_completion, s.event, label=lab)    ax = kmf.plot_survival_function(ax=ax)ax.set_xlabel("Months from trial start"); ax.set_ylabel("Proportion not yet completed")ax.set_xlim(0, 120)

In [ ]:
cph_cols = ["months_to_primary_completion","event","ip_strong","enrollment",            "is_phase3","n_locations","is_industry"]cx = surv[cph_cols].dropna()cph = CoxPHFitter().fit(cx, duration_col="months_to_primary_completion", event_col="event")cph.print_summary(decimals=4)print("\nConcordance index:", round(cph.concordance_index_, 4))# proportional hazards diagnosticcph.check_assumptions(cx, p_value_threshold=0.05, show_plots=False)

## 6. RQ4 — Moderation by therapeutic area

In [ ]:
# Therapeutic-area strata come from the MeSH mapping built in prep_06_mesh_areas.py
# (AACT browse_conditions ancestors -> top-level MeSH disease categories,
#  priority-assigned so strata are mutually exclusive). Already a column on the frame.
print(frame.therapeutic_area.value_counts().to_string())

from statsmodels.stats.proportion import proportion_confint
rows = []
for a, g_ in frame.groupby("therapeutic_area"):
    lo, hi = proportion_confint(int(g_.is_terminated.sum()), len(g_), method="wilson")
    rows.append({"area": a, "n": len(g_), "p_term": round(g_.is_terminated.mean(), 4),
                 "ci_low": round(lo, 4), "ci_high": round(hi, 4),
                 "margin": round((hi - lo) / 2, 4), "meets_737": len(g_) >= 737})
strata = pd.DataFrame(rows).sort_values("n", ascending=False)
display(strata)
assert strata.meets_737.all(), "a stratum falls below the RQ4 requirement of 737"
print("\nAll strata meet the n >= 737 requirement.")

In [ ]:
# moderated logistic regression: IP x therapeutic aream0 = smf.logit("is_terminated ~ ip_strong + C(therapeutic_area) + is_phase3 + is_industry",               data=frame).fit(disp=0)m1 = smf.logit("is_terminated ~ ip_strong * C(therapeutic_area) + is_phase3 + is_industry",               data=frame).fit(disp=0)lr_stat = 2 * (m1.llf - m0.llf)from scipy.stats import chi2 as chi2distdf_diff = m1.df_model - m0.df_modelprint(f"Likelihood ratio test: chi2 = {lr_stat:.3f}, df = {df_diff:.0f}, "      f"p = {chi2dist.sf(lr_stat, df_diff):.4f}")print(m1.summary())